# Data Summarization

## Basic Operations of Data Manipulations

You have learned several key operations that allow you to solve the vast majority of your data manipulation challenges:

* Pick observations by their values.
* Reorder the rows.
* Pick variables by their names.
* Create new variables with functions of existing variables.
* **Collapse many values down to a single summary.**

These can all be used in conjunction with groupby() which changes the scope of each function from operating on the entire dataset to operating on it group-by-group. 

In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
from nycflights13 import flights
flights.head()

## DataFrame with columns

- year,month,day
        Date of departure    
- dep_time,arr_time
        Actual departure and arrival times (format HHMM or HMM), local tz.
- sched_dep_time,sched_arr_time
        Scheduled departure and arrival times (format HHMM or HMM), local tz.    
- dep_delay,arr_delay
        Departure and arrival delays, in minutes. Negative times represent early departures/arrivals.
- hour,minute
        Time of scheduled departure broken into hour and minutes.
- carrier
        Two letter carrier abbreviation. See airlines() to get name
- tailnum
        Plane tail number
- flight
        Flight number
- origin,dest
        Origin and destination. See airports() for additional metadata.
- air_time
        Amount of time spent in the air, in minutes
- distance
        Distance between airports, in miles
- time_hour
        Scheduled date and hour of the flight as a date. Along with origin, can be used to join flights data to weather data.

In [ ]:
# Basic descriptive statistics for each column 
flights.describe()

In [ ]:
flights.describe(include='all')

In [ ]:
# Dimensions of the dataframe
flights.shape

In [ ]:
# Number of rows in the dataframe
len(flights)

In [ ]:
# Number of distinct values in a column.
flights['carrier'].nunique()

In [ ]:
# Count number of rows with each unique value of a variable
flights['carrier'].value_counts()

## Summary functions

Pandas provides a large set of summary functions that operate on different kinds of pandas objects (DataFrame columns, Series, GroupBy), and produce single values for each of the groups. When applied to a DataFrame, the result is returned as a pandas Series for each column. Examples:
- ``sum()`` Sum values of each object.
- ``count()`` Count non-NA/null values of each object.
- ``median()`` Median value of each object.
- ``quantile([0.25,0.75])`` Quantiles of each object.
- ``min()`` Minimum value in each object.
- ``max()`` Maximum value in each object.
- ``mean()`` Mean value of each object.
- ``var()`` Variance of each object.
- ``std()`` Standard deviation of each object.
- ``apply(function)`` Apply function to each object.

These summary functions can be applied to all the rows in the dataframe. 

In [ ]:
# Count the number of flights (rows)
flights['flight'].count()

In [ ]:
# Sum up the total distance of all flights. 
flights['distance'].sum()

In [ ]:
# Average/mean of arrival delay
flights['arr_delay'].mean()

In [ ]:
# Apply a function to multiple columns
flights[['distance','air_time']].max()

## Group by

These summary functions are not terribly useful unless we pair them with groupby(). This changes the unit of analysis from the complete dataset to individual groups. Then, when you use a summary function on a grouped data frame they’ll be automatically applied “by group”. 

In [ ]:
flights.groupby('carrier').size()

In [ ]:
flights.groupby(['carrier','flight']).size()

In [ ]:
flights.groupby(['year','month','day'])['arr_delay'].mean()

In [ ]:
flights.groupby(['year','month'])['arr_delay'].agg(['mean','std','min','max'])

What if we want to apply different summary functions to different columns? 

In [ ]:
flights.groupby('carrier').agg({'flight': 'size',
                                'distance': 'sum', 
                                'arr_delay': ['mean','std'],
                                'hour': lambda x: x.max()-x.min()
                               })

In [ ]:
# You can also create a function to include multiple aggregate functions on different columns. 
# In this way, you can give a name for each new column in the resulting dataframe. 
def f(x):
    d = {}
    d['flight_count'] = x['flight'].count()
    d['total_distance'] = x['distance'].sum()
    d['arr_delay_mean'] = x['arr_delay'].mean()
    d['arr_delay_std'] = x['arr_delay'].std()
    d['hour_range'] = x['hour'].max() - x['hour'].min()
    return pd.Series(d)

flights.groupby('carrier').apply(f)

## Combining multiple operations

Now let's put multiple operations we've learned together. 

### Q1. How many flights left before 5am on each day? (These usually indicate delayed flights from the previous day)

In [ ]:
# Write the code below: 


### Q2. For each destination from any of the three airports in NYC, explore the relationship between the distance and average delay. 

Follow the three steps:
- Group flights by destination and summarise to compute number of flights, average distance, and average arrival delay.
- Filter to remove noisy points and Honolulu airport (HNL), which is almost twice as far away as the next closest airport.
- Sort all rows by arrival delay.

In [ ]:
# Your code step by step: 


In [ ]:
# Put everything in one line of code: 


### Q3. Find the planes (identified by the tail number) that have the highest average arrival delays.

In [ ]:
# First, find the flights that are not cancelled. Save them in a separate dataframe named "not_cancelled"
# Assumption: flights that are not cancelled should have values for dep_delay and arr_delay. 


In [ ]:
# For each plane, find the average arrival delays and the flight count. 
# Save the results in a dataframe named "delays"


In [ ]:
# Explore the "delays" dataframe. 
# Try to create a scatter plot to show 'flight' vs. 'arr_delay'
# https://pandas.pydata.org/pandas-docs/version/0.25.0/reference/api/pandas.DataFrame.plot.scatter.html


**What is the range of average arrival delay for these planes?**

**Any outliers?**
 

## Operations within groups

Grouping is most useful in conjunction with aggregate functions. But you can also do other operations within groups:

In [ ]:
# Find the worst members of each group:

# Find the top three flights with the longest arr_delay every day. 
flights['rank_daily_delay'] = flights.groupby(['year', 'month', 'day'])['arr_delay'].rank(method='min',ascending=False)
flights[flights.rank_daily_delay<=3]

In [ ]:
# Find all groups bigger than a threshold:

# Find all flights that fly to the popular destinations that appear over 1000 times(flights). 
flights.groupby('dest').filter(lambda x: x['dest'].count()>1000)

In [ ]:
# Standardise to compute per group metrics:

# For all flights that arrived later than scheduled, 
# calculate the proportion of arrival delay among delayed flights to each destination
# display year, month, day, destination, flight, arr_delay, and proportion of arr_delay
flights['prop_delay'] = flights[flights.arr_delay>0].groupby('dest')['arr_delay'].transform(lambda x: x / x.sum())
flights[flights.arr_delay>0][['year', 'month', 'day', 'dest', 'flight', 'arr_delay', 'prop_delay']]    